In [ ]:
import os

# Trouve le dossier racine du projet en remontant depuis le dossier courant jusqu'à ce qu'on trouve 'data'
def find_project_root(current_path, target_folder="data"):
    path = current_path
    while True:
        if target_folder in os.listdir(path):
            return path
        new_path = os.path.dirname(path)
        if new_path == path:  # On est arrivé à la racine du disque sans trouver
            raise FileNotFoundError(f"Le dossier '{target_folder}' n'a pas été trouvé dans la hiérarchie des dossiers.")
        path = new_path

# Récupère le dossier courant
current_dir = os.getcwd()

# Trouve la racine du projet (dossier qui contient 'data')
project_root = find_project_root(current_dir, target_folder="data")

# Construit le chemin vers le dossier data/processed
data_processed_dir = os.path.join(project_root, "data", "processed")

# Chemin complet vers le fichier CSV
csv_path = os.path.join(data_processed_dir, "accidents_clean.csv")

print("Chemin absolu vers le fichier :", csv_path)

# Chargement du fichier
import pandas as pd
df = pd.read_csv(csv_path)
print(df.head())


Chemin absolu vers le fichier : /Users/alizeeblanchon/Documents/Data_Scientist/data_project/mai25_bds_accidents/data/processed/accidents_clean.csv
        Num_Acc  id_vehicule num_veh  catv  obs  obsm  choc  manv  motor  \
0  201900000001  138 306 524     B01     7    0     2     5    23      1   
1  201900000001  138 306 524     B01     7    0     2     5    23      1   
2  201900000001  138 306 525     A01    17    1     0     3    11      1   
3  201900000002  138 306 523     A01     7    4     0     1     0      1   
4  201900000003  138 306 520     A01     7    0     2     1     2      1   

   place  ...   nbv  prof  plan  surf  infra  situ  vma   age  \
0      2  ...  10.0     1     2     1      2     1   70  17.0   
1      1  ...  10.0     1     2     1      2     1   70  26.0   
2      1  ...  10.0     1     2     1      2     1   70  60.0   
3      1  ...   2.0     4     2     1      0     1   70  25.0   
4      1  ...   8.0     1     3     1      0     1   90  23.0   

     

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN

from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# === 1. Chargement des données


# === 2. Séparation X, y
X = df.drop(columns=['grav']).copy()
y = df['grav']

# === 3. Colonnes à encoder
# On suppose qu'il faut encoder 'dep' par TargetEncoder, les autres colonnes catégorielles par OneHotEncoder
# Sauf si tu connais précisément quelles colonnes sont catégorielles, je prends un exemple ici

# Identifie les colonnes catégorielles (hors 'dep'), par exemple
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if 'dep' in categorical_cols:
    categorical_cols.remove('dep')

# Colonnes numériques (à scaler)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes à one-hot encoder : toutes les catégorielles sauf 'dep'
cols_to_ohe = categorical_cols

# === 4. Préprocessing
preprocessor = ColumnTransformer(transformers=[
    ('ohe', SkPipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cols_to_ohe),
    ('target_enc', SkPipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('target', TargetEncoder())
    ]), ['dep']),
    ('num', StandardScaler(), numeric_cols)
], remainder='drop')

# === 5. Pipeline avec SMOTEENN + XGBoost
pipeline = ImbPipeline(steps=[
    ('preprocessing', preprocessor),
    ('sampling', SMOTEENN(random_state=42)),
    ('classifier', XGBClassifier(
        objective='multi:softmax',
        eval_metric='mlogloss',
        use_label_encoder=False,
        num_class=len(np.unique(y)),
        random_state=42
    ))
])

# === 6. Hyperparamètres pour GridSearch
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 6],
    'classifier__learning_rate': [0.1, 0.3],
    'classifier__subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv,
                           scoring='f1_weighted', n_jobs=-1, verbose=1)

grid_search.fit(X, y)

# === 7. Résultats
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X)

print("Meilleur score F1 pondéré (CV) :", grid_search.best_score_)
print("Meilleurs paramètres :", grid_search.best_params_)
print("\nClassification Report :\n", classification_report(y, y_pred))

# === 8. Matrice de confusion
cm = confusion_matrix(y, y_pred)
classes = sorted(y.unique())
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - XGBoost Optimisé")
plt.tight_layout()
plt.show()


Fitting 5 folds for each of 16 candidates, totalling 80 fits
